[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1g2-K0Ku2yW0gqhgTIemtgy7JNJyVstLF)

# Overview

With collaborative filtering, an application can find users with similar tastes and can look at items they like and combine them to create a ranked list of suggestions which is known as **user based recommendation**. 

Or we can also find items which are similar to each other and then suggest the items to users based on their past purchases which is known as **item based recommendation**. 

The first step in this technique is to find users with similar tastes or items which share similarity. 

There are various similarity models like** Cosine Similarity, Euclidean Distance Similarity and Pearson Correlation Similarity** which can be used to find similarity between users or items.

#Data

https://grouplens.org/datasets/movielens/

The data set is posted under the section: ***recommended for education and development*** and we will use the small version of the data set with 100,000 ratings

In [0]:
!wget http://files.grouplens.org/datasets/movielens/ml-latest-small.zip

--2018-06-17 20:27:52--  http://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.34.235
Connecting to files.grouplens.org (files.grouplens.org)|128.101.34.235|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 918269 (897K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 896.75K  3.86MB/s    in 0.2s    

2018-06-17 20:27:52 (3.86 MB/s) - ‘ml-latest-small.zip’ saved [918269/918269]



In [0]:
!unzip -o ml-latest-small.zip

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/movies.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/tags.csv  


In [0]:
import pandas as pd
import numpy as np

In [0]:
df_movies = pd.read_csv('ml-latest-small/movies.csv')
print(df_movies.shape)
df_movies.head()

(9125, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [0]:
df_ratings = pd.read_csv('ml-latest-small/ratings.csv')
print(df_ratings.shape)
df_ratings.head()

(100004, 4)


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


# Pivot Table
to create sparse user vectors with columns as different movies, rows as the userIds and value as the ratings

In [0]:
dataset_users = df_ratings.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)

In [0]:
dataset_users.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,...,0.0,0.0,0,0,0.0,0,0.0,0,0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,4.0,...,0.0,0.0,0,0,0.0,0,0.0,0,0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,...,0.0,0.0,0,0,0.0,0,0.0,0,0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,4.0,...,0.0,0.0,0,0,0.0,0,0.0,0,0,0
5,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0,0,0.0,...,0.0,0.0,0,0,0.0,0,0.0,0,0,0


# User Similarity

In [0]:
import sklearn.metrics as metrics

user_similarity_matrix = metrics.pairwise.cosine_similarity(dataset_users.as_matrix())
print(user_similarity_matrix.shape)
print(user_similarity_matrix)

(671, 671)
[[1.         0.         0.         ... 0.06291708 0.         0.01746565]
 [0.         1.         0.12429498 ... 0.02413984 0.17059464 0.1131753 ]
 [0.         0.12429498 1.         ... 0.08098382 0.13660585 0.17019275]
 ...
 [0.06291708 0.02413984 0.08098382 ... 1.         0.04260878 0.08520194]
 [0.         0.17059464 0.13660585 ... 0.04260878 1.         0.22867673]
 [0.01746565 0.1131753  0.17019275 ... 0.08520194 0.22867673 1.        ]]


**make all the diagonal elements as 0 – as those represent the correlation of each user to itself**

In [0]:
np.fill_diagonal(user_similarity_matrix, 0)
print(user_similarity_matrix)

[[0.         0.         0.         ... 0.06291708 0.         0.01746565]
 [0.         0.         0.12429498 ... 0.02413984 0.17059464 0.1131753 ]
 [0.         0.12429498 0.         ... 0.08098382 0.13660585 0.17019275]
 ...
 [0.06291708 0.02413984 0.08098382 ... 0.         0.04260878 0.08520194]
 [0.         0.17059464 0.13660585 ... 0.04260878 0.         0.22867673]
 [0.01746565 0.1131753  0.17019275 ... 0.08520194 0.22867673 0.        ]]


**The above gives the User Similarity Matrix – if we want to answer Qs like 'most similar users to userX', then we can make use of a DataFrame as shown below**

In [0]:
user_similarity_df = pd.DataFrame(user_similarity_matrix)
user_similarity_df.head()

,0,1,2,3,4,5,6,7,8,9,...,661,662,663,664,665,666,667,668,669,670
0,0.000000,0.000000,0.000000,0.074482,0.016818,0.000000,0.083884,0.000000,0.012843,0.000000,...,0.000000,0.000000,0.014474,0.043719,0.000000,0.000000,0.000000,0.062917,0.000000,0.017466
1,0.000000,0.000000,0.124295,0.118821,0.103646,0.000000,0.212985,0.113190,0.113333,0.043213,...,0.477306,0.063202,0.077745,0.164162,0.466281,0.425462,0.084646,0.024140,0.170595,0.113175
2,0.000000,0.124295,0.000000,0.081640,0.151531,0.060691,0.154714,0.249781,0.134475,0.114672,...,0.161205,0.064198,0.176134,0.158357,0.177098,0.124562,0.124911,0.080984,0.136606,0.170193
3,0.074482,0.118821,0.081640,0.000000,0.130649,0.079648,0.319745,0.191013,0.030417,0.137186,...,0.114319,0.047228,0.136579,0.254030,0.121905,0.088735,0.068483,0.104309,0.054512,0.211609
4,0.016818,0.103646,0.151531,0.130649,0.000000,0.063796,0.095888,0.165712,0.086616,0.032370,...,0.191029,0.021142,0.146173,0.224245,0.139721,0.058252,0.042926,0.038358,0.062642,0.225086


In [0]:
def top_similar_users(user_index, count=5):
  print("Top %d users similar to user%d:" % (count, user_index))
  print("-"*30)
  user_column = user_similarity_df.iloc[:, user_index]
  sorted_user_column = user_column.sort_values(ascending=False)
  print(sorted_user_column[0:count])
  print("-"*30)

In [0]:
top_similar_users(1)

Top 5 users similar to user1:
------------------------------
337    0.581528
368    0.580742
150    0.573097
399    0.571252
384    0.565113
Name: 1, dtype: float64
------------------------------


In [0]:
top_similar_users(200)

Top 5 users similar to user200:
------------------------------
294    0.456824
561    0.424542
409    0.404349
281    0.382866
354    0.379410
Name: 200, dtype: float64
------------------------------


### Examine 2 similar users and examine if the above similarity measure indeed worked

Le'ts pick user with index 200 and 294 to examine (Note: `userId` in the `df_ratings` dataframe is 1-indexed)

In [0]:
def examine_similar_users(user1, user2):
  # drop timestamp as we don't need it in this context
  ratings_without_timestamp = df_ratings.drop(['timestamp'], axis=1)
  
  df_ratings1 = ratings_without_timestamp[ratings_without_timestamp['userId']==user1]  
  df_ratings2 = ratings_without_timestamp[ratings_without_timestamp['userId']==user2]  
  df_ratings_common = df_ratings1.merge(df_ratings2, on='movieId', how='inner')
  
  # we also want movie details like names etc, so we need to merge again with the df_movies dataframe
  df_movies_common = df_ratings_common.merge(df_movies, on='movieId')
  return df_movies_common.head()

In [0]:
examine_similar_users(201, 295) # userId is 1-indexed

,userId_x,movieId,rating_x,userId_y,rating_y,title,genres
0,201,6,5.0,295,4.5,Heat (1995),Action|Crime|Thriller
1,201,47,5.0,295,4.5,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
2,201,50,5.0,295,4.5,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
3,201,110,5.0,295,4.0,Braveheart (1995),Action|Drama|War
4,201,150,5.0,295,4.5,Apollo 13 (1995),Adventure|Drama|IMAX


**Observation:** User_201 and User_295 which had a high user similarity, does seem to have rated movies in similar fashion

# Item Similarity

We want to repeat the analogous steps as above, just this time we want to use the **Pearson Correlation Similarity Coefficient**

In [0]:
dataset_movies = df_ratings.pivot_table(index='movieId', columns='userId', values='rating', fill_value=0)

In [0]:
dataset_movies

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0,0.0,0,0.0,0.0,3,0.0,4,0,...,0,4.0,3.5,0,0,0,0,0,4,5.0
2,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,5,0.0,0.0,3,0,0,0,0,0,0.0
3,0.0,0,0.0,0,4.0,0.0,0,0.0,0,0,...,0,0.0,0.0,3,0,0,0,0,0,0.0
4,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0,0,0,0,0,0,0.0
5,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,3,0,0,0,0,0,0.0
6,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,4.0,0,5,4,0,0,0,0.0
7,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0,0,0,0,0,0,0.0
8,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0,0,0,0,0,0,0.0
9,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,...,0,0.0,0.0,0,0,0,0,0,0,0.0


We could either use `metrics.pairwise.cosine_similarity` as earlier – or we can try a different approach here

---


**Pandas has a `corr()` function which takes in `method` as an argument, to which we can pass `'pearson'`**

The `corr()` function computes the correlation between the columns – and we want the correlation between the movies – so we ll use the `dataset_users` dataframe which had movies as columns

In [0]:
item_similarity_matrix = dataset_users.corr(method='pearson').as_matrix()
print(item_similarity_matrix.shape)
print(item_similarity_matrix)

(9066, 9066)
[[ 1.          0.22374218  0.18326579 ... -0.0281574  -0.0281574
   0.04097762]
 [ 0.22374218  1.          0.12379014 ... -0.01619963 -0.01619963
  -0.01619963]
 [ 0.18326579  0.12379014  1.         ... -0.01122147 -0.01122147
  -0.01122147]
 ...
 [-0.0281574  -0.01619963 -0.01122147 ...  1.          1.
  -0.00149254]
 [-0.0281574  -0.01619963 -0.01122147 ...  1.          1.
  -0.00149254]
 [ 0.04097762 -0.01619963 -0.01122147 ... -0.00149254 -0.00149254
   1.        ]]


In [0]:
np.fill_diagonal(item_similarity_matrix, 0)
print(item_similarity_matrix)

[[ 0.          0.22374218  0.18326579 ... -0.0281574  -0.0281574
   0.04097762]
 [ 0.22374218  0.          0.12379014 ... -0.01619963 -0.01619963
  -0.01619963]
 [ 0.18326579  0.12379014  0.         ... -0.01122147 -0.01122147
  -0.01122147]
 ...
 [-0.0281574  -0.01619963 -0.01122147 ...  0.          1.
  -0.00149254]
 [-0.0281574  -0.01619963 -0.01122147 ...  1.          0.
  -0.00149254]
 [ 0.04097762 -0.01619963 -0.01122147 ... -0.00149254 -0.00149254
   0.        ]]


In [0]:
movie_similarity_df = pd.DataFrame(item_similarity_matrix)
movie_similarity_df.head()

,0,1,2,3,4,5,6,7,8,9,...,9056,9057,9058,9059,9060,9061,9062,9063,9064,9065
0,0.000000,0.223742,0.183266,0.071055,0.105076,0.201503,0.156075,0.019379,0.023699,0.089163,...,0.040978,0.011348,0.070607,0.070607,0.070607,0.070607,0.070607,-0.028157,-0.028157,0.040978
1,0.223742,0.000000,0.123790,0.125014,0.193144,0.085889,0.117211,0.209299,0.053810,0.306685,...,-0.016200,0.043525,0.058457,0.073388,0.073388,0.133113,0.058457,-0.016200,-0.016200,-0.016200
2,0.183266,0.123790,0.000000,0.147771,0.317911,0.158071,0.390331,0.109818,0.274638,0.086065,...,-0.011221,-0.011221,-0.011221,0.109898,0.109898,-0.011221,-0.011221,-0.011221,-0.011221,-0.011221
3,0.071055,0.125014,0.147771,0.000000,0.150562,0.024466,0.156876,0.496859,0.238193,0.063511,...,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073,-0.005073
4,0.105076,0.193144,0.317911,0.150562,0.000000,0.186936,0.339605,0.179371,0.339402,0.150292,...,-0.011165,0.173054,-0.011165,0.111648,0.111648,-0.011165,-0.011165,-0.011165,-0.011165,-0.011165


In [0]:
def top_similar_movies(movieId, count=5):
  index = movieId-1 # movies are 1-indexed
  print("Top %d movies similar to movie %d:" % (count, movieId))
  print("-"*100)
  df_movie_similarity = df_movies.copy()
  df_movie_similarity['similarity'] = movie_similarity_df.iloc[:, index]
  sorted_df_movie_similarity = df_movie_similarity.sort_values(['similarity'], ascending=False)
  return sorted_df_movie_similarity[0:count]

In [0]:
top_similar_movies(6)

Top 5 movies similar to movie 6:
----------------------------------------------------------------------------------------------------


,movieId,title,genres,similarity
615,733,"Rock, The (1996)",Action|Adventure|Thriller,0.430485
24,25,Leaving Las Vegas (1995),Drama|Romance,0.421901
650,786,Eraser (1996),Action|Drama|Thriller,0.398260
87,95,Broken Arrow (1996),Action|Adventure|Thriller,0.375565
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,0.367041


In [0]:
top_similar_movies(42)

Top 5 movies similar to movie 42:
----------------------------------------------------------------------------------------------------


,movieId,title,genres,similarity
401,452,Widows' Peak (1994),Drama,0.595277
280,314,"Secret of Roan Inish, The (1994)",Children|Drama|Fantasy|Mystery,0.589898
1943,2434,Down in the Delta (1998),Drama,0.527189
339,375,Safe Passage (1994),Drama,0.481002
1137,1399,Marvin's Room (1996),Drama,0.465627


# Challenges

1.   One immediately visible problem is that how do we handle **new users**, for which we have no data at all – we could potentially throw random stuff 1st, and then try to make a decision – but it's going to result in a poor experience for the user until the model starts to perform for that user
2.   The above will happen for **new/unrated movies** also – we have to wait until enough users have rated a particular movie, before it can be of any use in the algorithm
3.   Similar to above, this could result in 1 of the common criticisms of Recommender Systems – the **long tail items never get picked** up as only the popular movies are watched/rated, and only those are further recommended to other users

